# Seed, Don't Spread: reproducing the paper

Companion notebook for the Gemma 4 Developer Agent Paper Track writeup.
Code: https://github.com/shineyrbl-byte/seed-and-spread (Apache-2.0)

This notebook:
1. bare-clones the 12 SWE-bench Lite repositories and builds a code graph for every task snapshot (no checkouts; standard-library `ast`),
2. re-runs the **no-LLM ranker benchmark** (Table 1) on Kaggle CPU,
3. loads the committed per-task **Gemma 4 E2B / E4B** outputs (those were produced on an 8 GB Apple laptop with MLX, which Kaggle cannot run) and regenerates every table and figure from them.

Requires *Internet on* (Settings → Internet). Set `LIMIT = 0` for the full 300 tasks (~45 min CPU) or a small number for a quick check.

In [ ]:
LIMIT = 60   # 0 = all 300 SWE-bench Lite test tasks
!git clone -q https://github.com/shineyrbl-byte/seed-and-spread.git
%cd seed-and-spread
!pip -q install rank-bm25==0.2.2 datasets

In [ ]:
!bash scripts/clone_repos.sh && du -sh cache/repos

In [ ]:
import os
os.environ["SS_LIMIT"] = str(LIMIT)
!python scripts/build_snapshots.py | tail -3

## Ranker benchmark (no LLM)
Recomputes file-ranking accuracy for every ranker on the snapshots built above. Committed full-run results are kept aside so we can compare.

In [ ]:
!cp results/rankers.jsonl results/rankers_full.jsonl
!sed -i 's/count = gemma_counter()/count = None/; s/render(g, ranked, b, layout, count=count)/render(g, ranked, b, layout)/' scripts/eval_rankers.py
!SS_TAG=kaggle python scripts/eval_rankers.py | tail -8

In [ ]:
import json
fresh = {r["id"]: r["rank"] for r in map(json.loads, open("results/rankers_kaggle.jsonl"))}
full = {r["id"]: r["rank"] for r in map(json.loads, open("results/rankers_full.jsonl"))}
for r in map(json.loads, open("results/rankers_seedfuse.jsonl")):
    full[r["id"]].update(r["rank"])
same = sum(all(fresh[i][k] == full[i][k] for k in fresh[i]) for i in fresh)
print(f"tasks re-ranked here: {len(fresh)}; identical ranks to the committed run: {same}/{len(fresh)}")

## All paper tables and figures from committed results
`analyze.py` produces every number in the writeup (bootstrap CIs, paired tests, per-repo breakdown, variant grids, Gemma 4 tables).

In [ ]:
!cp results/rankers_full.jsonl results/rankers.jsonl
!python scripts/analyze.py

In [ ]:
!pip -q install matplotlib
!python scripts/figures.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("paper/fig_*.png")):
    display(Image(f))

## Running Gemma 4 yourself
On an Apple-silicon Mac (8 GB is enough):
```
uv sync
uv run python scripts/eval_llm.py mlx-community/gemma-4-e2b-it-4bit 100 none/none/0,seed+bm25/sigs/2000
```
Each 100-task condition takes about 10–20 minutes.